# Experiment 8
## Variational Autoencoder (VAE)
**Aim:** Build an encoder-decoder architecture for a VAE and train it on MNIST.

### Step 1 – Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

### Step 2 – Load MNIST Dataset

In [ ]:
transform    = transforms.Compose([transforms.ToTensor()])
train_data   = datasets.MNIST('./data', train=True,  download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
print('Samples:', len(train_data))

### Step 3 – Define the VAE Architecture

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()
        # Encoder
        self.enc_fc1 = nn.Linear(784, 256)
        self.enc_mu  = nn.Linear(256, latent_dim)
        self.enc_var = nn.Linear(256, latent_dim)
        # Decoder
        self.dec_fc1 = nn.Linear(latent_dim, 256)
        self.dec_fc2 = nn.Linear(256, 784)

    def encode(self, x):
        h   = F.relu(self.enc_fc1(x))
        return self.enc_mu(h), self.enc_var(h)

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std                 # reparameterization trick

    def decode(self, z):
        h = F.relu(self.dec_fc1(z))
        return torch.sigmoid(self.dec_fc2(h))

    def forward(self, x):
        x = x.view(-1, 784)
        mu, log_var = self.encode(x)
        z    = self.reparameterize(mu, log_var)
        recon = self.decode(z)
        return recon, mu, log_var

### Step 4 – Define VAE Loss (Reconstruction + KL Divergence)

In [ ]:
def vae_loss(recon_x, x, mu, log_var):
    recon_loss = F.binary_cross_entropy(recon_x, x.view(-1, 784), reduction='sum')
    kl_div     = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
    return recon_loss + kl_div

### Step 5 – Train the VAE

In [ ]:
vae       = VAE(latent_dim=16)
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)
losses    = []
for epoch in range(5):
    total_loss = 0
    for imgs, _ in train_loader:
        recon, mu, log_var = vae(imgs)
        loss = vae_loss(recon, imgs, mu, log_var)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item()
    avg = total_loss / len(train_data)
    losses.append(avg)
    print(f'Epoch {epoch+1}/5 | Loss: {avg:.4f}')

### Step 6 – Reconstruct and Generate New Images

In [ ]:
vae.eval()
with torch.no_grad():
    sample_imgs = next(iter(train_loader))[0][:8]
    recon, _, _ = vae(sample_imgs)

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i in range(8):
    axes[0, i].imshow(sample_imgs[i].squeeze(), cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(recon[i].view(28,28).numpy(), cmap='gray'); axes[1, i].axis('off')
axes[0,0].set_ylabel('Original', fontsize=9); axes[1,0].set_ylabel('Reconstructed', fontsize=9)
plt.suptitle('VAE Reconstructions'); plt.show()

### Result
A Variational Autoencoder was built with separate encoder and decoder networks. The model learned to compress and reconstruct MNIST digits using the reparameterization trick.